In [ ]:
# CELL 1 — Restore and verify all four baseline models.
# No training. Does not recreate or change experimental splits.

from pathlib import Path, PurePosixPath
import hashlib
import json
import os
import shutil
import tempfile
import zipfile

INPUT = Path("/kaggle/input")
ROOT = Path("/kaggle/working/wheat_rice_journal")
ROOT.parent.mkdir(parents=True, exist_ok=True)

RUNS = [
    f"{crop}_{model}_baseline_seed42"
    for crop in ("wheat", "rice")
    for model in ("efficientnet_b0", "mobilenet_v2")
]

REQUIRED = [
    "splits/split_config.json",
    "data_audit/class_maps.json",
]
for run in RUNS:
    REQUIRED.extend(
        f"runs/{run}/{name}"
        for name in (
            "best_model.pt",
            "latest_checkpoint.pt",
            "config.json",
            "validation_summary.json",
            "training_log.csv",
        )
    )


def file_hash(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def restore_verified_backup():
    print("CELL 1 STARTED — locating verified backup...", flush=True)
    source = None
    manifest_path = None
    staging = None
    prefix = "wheat_rice_journal/"

    try:
        # Locate an already-extracted backup.
        for candidate_manifest in sorted(INPUT.rglob("BACKUP_MANIFEST.json")):
            candidate = candidate_manifest.parent / "wheat_rice_journal"
            if all((candidate / name).is_file() for name in REQUIRED):
                source = candidate
                manifest_path = candidate_manifest
                break

        # Otherwise locate and extract the matching ZIP.
        if source is None:
            for archive_path in sorted(INPUT.rglob("*.zip")):
                with zipfile.ZipFile(archive_path) as archive:
                    names = set(archive.namelist())
                    if (
                        "BACKUP_MANIFEST.json" not in names
                        or not all(prefix + name in names for name in REQUIRED)
                    ):
                        continue

                    print("Found backup ZIP:", archive_path, flush=True)
                    staging = tempfile.TemporaryDirectory(
                        prefix="four_model_restore_", dir="/kaggle/working"
                    )
                    stage_root = Path(staging.name)

                    for member in archive.infolist():
                        if member.is_dir():
                            continue
                        if not (
                            member.filename.startswith(prefix)
                            or member.filename == "BACKUP_MANIFEST.json"
                        ):
                            continue

                        relative = PurePosixPath(member.filename)
                        if relative.is_absolute() or ".." in relative.parts:
                            raise RuntimeError("Invalid archive path.")

                        target = stage_root.joinpath(*relative.parts)
                        target.parent.mkdir(parents=True, exist_ok=True)
                        with archive.open(member) as src, target.open("wb") as dst:
                            shutil.copyfileobj(src, dst)

                    source = stage_root / "wheat_rice_journal"
                    manifest_path = stage_root / "BACKUP_MANIFEST.json"
                    break

        if source is None:
            raise FileNotFoundError(
                "Complete four-model backup with BACKUP_MANIFEST.json "
                "not found in attached inputs."
            )

        print("Backup located:", source, flush=True)
        manifest = json.loads(manifest_path.read_text())
        entries = manifest.get("sha256", {})

        if not all(prefix + name in entries for name in REQUIRED):
            raise RuntimeError("Backup manifest is incomplete.")

        print("Verifying all backup files...", flush=True)
        verified = {}

        for member, expected in entries.items():
            relative = PurePosixPath(member)
            if (
                relative.is_absolute()
                or ".." in relative.parts
                or len(relative.parts) < 2
                or relative.parts[0] != "wheat_rice_journal"
            ):
                raise RuntimeError(f"Invalid manifest entry: {member}")

            local_path = Path(*relative.parts[1:])
            source_file = source / local_path
            if not source_file.is_file() or file_hash(source_file) != expected:
                raise RuntimeError(f"Backup hash mismatch: {member}")
            verified[local_path] = expected

        config = json.loads(
            (source / "splits" / "split_config.json").read_text()
        )
        hashes = config.get("split_sha256", {})
        expected_splits = {
            f"{crop}_{part}.csv"
            for crop in ("wheat", "rice")
            for part in ("train", "validation", "test")
        }

        if not expected_splits.issubset(hashes):
            raise RuntimeError("Split configuration incomplete.")

        for name, expected in hashes.items():
            path = source / "splits" / name
            if not path.is_file() or file_hash(path) != expected:
                raise RuntimeError(f"Split verification failed: {name}")

        maps = json.loads(
            (source / "data_audit" / "class_maps.json").read_text()
        )
        for run in RUNS:
            crop = run.split("_")[0]
            run_config = json.loads(
                (source / "runs" / run / "config.json").read_text()
            )
            if (
                run_config.get("split_sha256") != hashes
                or run_config.get("class_map") != maps[crop]
            ):
                raise RuntimeError(f"Model configuration mismatch: {run}")

        # Stop before copying if an existing destination file differs.
        for relative, expected in verified.items():
            target = ROOT / relative
            if target.exists() and (
                not target.is_file() or file_hash(target) != expected
            ):
                raise RuntimeError(
                    f"Existing file differs: {relative}. Nothing overwritten."
                )

        restored = 0
        for relative, expected in verified.items():
            target = ROOT / relative
            if target.is_file():
                continue

            target.parent.mkdir(parents=True, exist_ok=True)
            temporary = target.with_name(target.name + ".restore_tmp")
            shutil.copyfile(source / relative, temporary)
            if file_hash(temporary) != expected:
                raise RuntimeError(f"Copy verification failed: {relative}")
            os.replace(temporary, target)
            restored += 1

        print(f"\nRESTORE COMPLETE — {restored} files restored.")
        print("All backup hashes and six split hashes verified.")

        for run in RUNS:
            summary = json.loads(
                (ROOT / "runs" / run / "validation_summary.json").read_text()
            )
            print(
                f"{run}: PRESENT | "
                f"validation macro-F1={summary['val_macro_f1']:.6f}"
            )

        import torch
        if not torch.cuda.is_available():
            raise RuntimeError(
                "Files restored, but GPU unavailable. "
                "Select a GPU accelerator before training."
            )

        print("GPU:", torch.cuda.get_device_name(0))
        print("CELL 1 COMPLETE — READY FOR DEGRADATION TRAINING")

    finally:
        if staging is not None:
            staging.cleanup()


restore_verified_backup()

In [ ]:
# CELL 2 — Degradation-aware EfficientNet-B0 + verified backup
# Requires Cell 1. Trains wheat, then rice.
# Baseline models and fixed splits are preserved.
# No test-set evaluation.

print("CELL 2 STARTED", flush=True)

from pathlib import Path
from io import BytesIO
import gc
import hashlib
import json
import os
import random
import socket
import time
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from IPython.display import display

ROOT = Path("/kaggle/working/wheat_rice_journal")
BACKUP = ROOT.parent / "wheat_rice_with_degradation_verified.zip"

SEED = 42
EPOCHS = 15
PATIENCE = 4
BATCH_SIZE = 32
WORKERS = 2
LR = 3e-4
WEIGHT_DECAY = 1e-4

if not torch.cuda.is_available():
    raise RuntimeError("GPU unavailable.")
DEVICE = torch.device("cuda:0")


def digest(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def save_json(value, path):
    path = Path(path)
    temp = path.with_name(path.name + ".tmp")
    temp.write_text(json.dumps(value, indent=2))
    os.replace(temp, path)


def save_csv(value, path):
    path = Path(path)
    temp = path.with_name(path.name + ".tmp")
    pd.DataFrame(value).to_csv(temp, index=False)
    os.replace(temp, path)


def save_torch(value, path):
    path = Path(path)
    temp = path.with_name(path.name + ".tmp")
    torch.save(value, temp)
    os.replace(temp, path)


def cpu_state(model):
    return {
        k: v.detach().cpu().clone()
        for k, v in model.state_dict().items()
    }


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def seed_worker(worker_id):
    seed = torch.initial_seed() % (2**32)
    random.seed(seed)
    np.random.seed(seed)


def backup(stage):
    print(f"Creating backup: {stage}", flush=True)
    temp = BACKUP.with_suffix(".zip.tmp")
    manifest = {}
    with zipfile.ZipFile(
        temp, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=1
    ) as archive:
        for path in sorted(ROOT.rglob("*")):
            if not path.is_file() or path.name.endswith(".tmp"):
                continue
            name = path.relative_to(ROOT.parent).as_posix()
            manifest[name] = digest(path)
            archive.write(path, name)
        archive.writestr(
            "BACKUP_MANIFEST.json",
            json.dumps({"stage": stage, "sha256": manifest}, indent=2)
        )

    with zipfile.ZipFile(temp) as archive:
        for name, expected in manifest.items():
            h = hashlib.sha256()
            with archive.open(name) as stream:
                for block in iter(lambda: stream.read(1024 * 1024), b""):
                    h.update(block)
            if h.hexdigest() != expected:
                raise RuntimeError(f"Backup mismatch: {name}")

    os.replace(temp, BACKUP)
    print(
        f"BACKUP VERIFIED: {BACKUP.name} "
        f"({BACKUP.stat().st_size / 1024**2:.1f} MB)",
        flush=True
    )


class RandomJPEG:
    def __init__(self, probability=0.3, quality_min=40, quality_max=90):
        self.probability = probability
        self.quality_min = quality_min
        self.quality_max = quality_max

    def __call__(self, image):
        if random.random() >= self.probability:
            return image
        quality = random.randint(self.quality_min, self.quality_max)
        with BytesIO() as buffer:
            image.save(buffer, format="JPEG", quality=quality)
            buffer.seek(0)
            with Image.open(buffer) as decoded:
                return decoded.convert("RGB").copy()

    def __repr__(self):
        return (
            f"RandomJPEG(probability={self.probability}, "
            f"quality_min={self.quality_min}, "
            f"quality_max={self.quality_max})"
        )


train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.RandomApply([
        transforms.ColorJitter(brightness=(0.7, 1.3))
    ], p=0.3),
    transforms.RandomApply([
        transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 1.5))
    ], p=0.3),
    RandomJPEG(0.3, 40, 90),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
    ),
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
    ),
])

config_path = ROOT / "splits/split_config.json"
maps_path = ROOT / "data_audit/class_maps.json"
if not config_path.is_file() or not maps_path.is_file():
    raise RuntimeError("Cell 1 must restore the backup first.")

split_config = json.loads(config_path.read_text())
class_maps = json.loads(maps_path.read_text())
hashes = split_config["split_sha256"]

required_names = {
    f"{crop}_{part}.csv"
    for crop in ("wheat", "rice")
    for part in ("train", "validation", "test")
}
if not required_names.issubset(hashes):
    raise RuntimeError("Incomplete split configuration.")

for name, expected in hashes.items():
    path = ROOT / "splits" / name
    if not path.is_file() or digest(path) != expected:
        raise RuntimeError(f"Split verification failed: {name}")

# Preserve fingerprints of all baseline checkpoints.
baseline_hashes = {}
for crop in ("wheat", "rice"):
    for architecture in ("efficientnet_b0", "mobilenet_v2"):
        folder = ROOT / "runs" / f"{crop}_{architecture}_baseline_seed42"
        config = json.loads((folder / "config.json").read_text())
        if (
            config.get("split_sha256") != hashes
            or config.get("class_map") != class_maps[crop]
        ):
            raise RuntimeError(f"Baseline configuration mismatch: {folder}")
        for name in ("best_model.pt", "latest_checkpoint.pt"):
            path = folder / name
            baseline_hashes[path] = digest(path)

frames = {}
directory_cache = {}

for crop in ("wheat", "rice"):
    frames[crop] = {}
    mapping = class_maps[crop]
    if sorted(mapping.values()) != list(range(len(mapping))):
        raise RuntimeError(f"Invalid class IDs: {crop}")

    for part in ("train", "validation"):
        frame = pd.read_csv(ROOT / "splits" / f"{crop}_{part}.csv")
        columns = ["path", "label", "label_id", "group_id"]
        if not set(columns).issubset(frame.columns):
            raise RuntimeError(f"Missing columns: {crop}/{part}")
        if frame.empty or frame[columns].isna().any().any():
            raise RuntimeError(f"Empty/incomplete inventory: {crop}/{part}")

        mapped = frame["label"].map(mapping)
        if (
            frame["path"].duplicated().any()
            or mapped.isna().any()
            or not np.array_equal(mapped.to_numpy(), frame["label_id"].to_numpy())
            or set(frame["label"]) != set(mapping)
        ):
            raise RuntimeError(f"Invalid inventory: {crop}/{part}")

        print(f"Checking {crop}/{part}: {len(frame):,} images", flush=True)
        grouped = {}
        for raw_path in frame["path"]:
            path = Path(raw_path)
            grouped.setdefault(path.parent, set()).add(path.name)

        for parent, names in grouped.items():
            if parent not in directory_cache:
                with os.scandir(parent) as entries:
                    directory_cache[parent] = {entry.name for entry in entries}
            missing = names - directory_cache[parent]
            if missing:
                raise FileNotFoundError(parent / sorted(missing)[0])

        frames[crop][part] = frame

    for column in ("path", "group_id"):
        if (
            set(frames[crop]["train"][column])
            & set(frames[crop]["validation"][column])
        ):
            raise RuntimeError(f"Train/validation overlap: {crop}/{column}")

(ROOT / "metrics").mkdir(parents=True, exist_ok=True)
print("Preflight passed. GPU:", torch.cuda.get_device_name(0), flush=True)


class CropDataset(Dataset):
    def __init__(self, frame, transform):
        self.paths = frame["path"].tolist()
        self.labels = frame["label_id"].astype(int).tolist()
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        path = self.paths[index]
        try:
            with Image.open(path) as source:
                image = ImageOps.exif_transpose(source).convert("RGB")
                image = self.transform(image)
        except Exception as exc:
            raise RuntimeError(f"Image loading failed: {path}") from exc
        return image, self.labels[index]


def plot_history(history, folder):
    data = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for col, label in (
        ("train_loss", "Training"), ("val_loss", "Validation")
    ):
        axes[0].plot(data["epoch"], data[col], label=label)
    axes[0].set(xlabel="Epoch", ylabel="Cross-entropy loss")

    for col, label in (
        ("train_accuracy", "Training accuracy"),
        ("val_accuracy", "Validation accuracy"),
        ("val_macro_f1", "Validation macro-F1"),
    ):
        axes[1].plot(data["epoch"], data[col], label=label)
    axes[1].set(xlabel="Epoch", ylabel="Score", ylim=(0, 1))
    for axis in axes:
        axis.legend()
        axis.grid(alpha=0.2)
    fig.tight_layout()
    for extension in ("png", "pdf"):
        fig.savefig(
            folder / f"learning_curves.{extension}",
            dpi=300, bbox_inches="tight"
        )
    plt.close(fig)


def train_crop(crop):
    folder = ROOT / "runs" / f"{crop}_efficientnet_b0_degradation_seed42"
    folder.mkdir(parents=True, exist_ok=True)
    latest = folder / "latest_checkpoint.pt"
    nclasses = len(class_maps[crop])

    config = {
        "crop": crop,
        "architecture": "torchvision_efficientnet_b0",
        "experiment": "degradation",
        "pretrained_weights": "EfficientNet_B0_Weights.IMAGENET1K_V1",
        "seed": SEED,
        "max_epochs": EPOCHS,
        "patience": PATIENCE,
        "batch_size": BATCH_SIZE,
        "learning_rate": LR,
        "weight_decay": WEIGHT_DECAY,
        "optimizer": "AdamW",
        "scheduler": "CosineAnnealingLR",
        "scheduler_eta_min": 1e-6,
        "loss": "unweighted_cross_entropy",
        "selection_metric": "clean_validation_macro_f1",
        "class_map": class_maps[crop],
        "split_sha256": hashes,
        "train_transform": repr(train_transform),
        "eval_transform": repr(eval_transform),
        "num_workers": WORKERS,
        "amp": True,
        "gradient_clip_norm": 5.0,
        "torch_version": str(torch.__version__),
        "gpu": torch.cuda.get_device_name(0),
    }

    checkpoint = None
    if latest.is_file():
        # Load only checkpoints from your own experiment.
        checkpoint = torch.load(
            latest, map_location="cpu", weights_only=False
        )
        if checkpoint["config"] != config:
            raise RuntimeError(f"{crop}: resume configuration differs.")
    elif (folder / "best_model.pt").exists():
        raise RuntimeError(f"{crop}: best model exists but resume file is missing.")

    history = checkpoint["history"] if checkpoint else []
    best_score = checkpoint["best_score"] if checkpoint else -1.0
    best_epoch = checkpoint["best_epoch"] if checkpoint else 0
    best_weights = checkpoint["best_weights"] if checkpoint else None
    bad_epochs = checkpoint["bad_epochs"] if checkpoint else 0
    start = checkpoint["epoch"] + 1 if checkpoint else 1

    save_json(config, folder / "config.json")
    print(f"\n{crop.upper()} DEGRADATION — starting at epoch {start}", flush=True)

    if start <= EPOCHS and bad_epochs < PATIENCE:
        seed_all(SEED)
        old_timeout = socket.getdefaulttimeout()
        print("Loading EfficientNet-B0...", flush=True)
        try:
            socket.setdefaulttimeout(60)
            model = models.efficientnet_b0(
                weights=None if checkpoint else
                models.EfficientNet_B0_Weights.IMAGENET1K_V1
            )
        finally:
            socket.setdefaulttimeout(old_timeout)

        model.classifier[1] = nn.Linear(model.classifier[1].in_features, nclasses)
        model.to(DEVICE)
        optimizer = torch.optim.AdamW(
            model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=EPOCHS, eta_min=1e-6
        )
        scaler = torch.amp.GradScaler("cuda")
        criterion = nn.CrossEntropyLoss()

        if checkpoint:
            model.load_state_dict(checkpoint["model"])
            optimizer.load_state_dict(checkpoint["optimizer"])
            for state in optimizer.state.values():
                for key in ("exp_avg", "exp_avg_sq", "max_exp_avg_sq"):
                    if key in state:
                        state[key] = state[key].to(DEVICE)
            scheduler.load_state_dict(checkpoint["scheduler"])
            scaler.load_state_dict(checkpoint["scaler"])
        checkpoint = None

        train_gen, val_gen = torch.Generator(), torch.Generator()
        loaders = {}
        for part, transform, generator in (
            ("train", train_transform, train_gen),
            ("validation", eval_transform, val_gen),
        ):
            loaders[part] = DataLoader(
                CropDataset(frames[crop][part], transform),
                batch_size=BATCH_SIZE,
                shuffle=(part == "train"),
                num_workers=WORKERS,
                pin_memory=True,
                persistent_workers=False,
                worker_init_fn=seed_worker,
                generator=generator,
                timeout=120,
            )

        for epoch in range(start, EPOCHS + 1):
            seed_all(SEED + epoch)
            train_gen.manual_seed(SEED + epoch)
            val_gen.manual_seed(SEED + epoch + 10000)
            started = time.perf_counter()
            lr_used = optimizer.param_groups[0]["lr"]
            model.train()
            loss_sum, correct, count = 0.0, 0, 0
            print(f"{crop} | epoch {epoch:02d} | loading batches...", flush=True)

            for batch, (images, targets) in enumerate(loaders["train"], 1):
                images = images.to(DEVICE, non_blocking=True)
                targets = targets.to(DEVICE, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast("cuda", dtype=torch.float16):
                    logits = model(images)
                    loss = criterion(logits, targets)
                if not torch.isfinite(loss).item():
                    raise RuntimeError("Non-finite training loss.")

                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                scaler.step(optimizer)
                scaler.update()

                loss_sum += loss.detach().item() * targets.size(0)
                correct += (logits.detach().argmax(1) == targets).sum().item()
                count += targets.size(0)
                if batch == 1 or batch % 50 == 0:
                    print(
                        f"{crop} | epoch {epoch:02d} | "
                        f"batch {batch}/{len(loaders['train'])}", flush=True
                    )

            print("Clean validation...", flush=True)
            model.eval()
            val_loss = 0.0
            truth, predictions = [], []
            with torch.inference_mode():
                for images, targets in loaders["validation"]:
                    images = images.to(DEVICE, non_blocking=True)
                    targets = targets.to(DEVICE, non_blocking=True)
                    with torch.autocast("cuda", dtype=torch.float16):
                        logits = model(images)
                        loss = criterion(logits, targets)
                    if not torch.isfinite(loss).item():
                        raise RuntimeError("Non-finite validation loss.")
                    val_loss += loss.item() * targets.size(0)
                    truth.extend(targets.cpu().tolist())
                    predictions.extend(logits.argmax(1).cpu().tolist())

            if (
                count != len(frames[crop]["train"])
                or len(truth) != len(frames[crop]["validation"])
            ):
                raise RuntimeError("Processed image count mismatch.")

            score = float(f1_score(
                truth, predictions, labels=list(range(nclasses)),
                average="macro", zero_division=0
            ))
            improved = score > best_score
            if improved:
                best_score, best_epoch = score, epoch
                best_weights = cpu_state(model)
                bad_epochs = 0
            else:
                bad_epochs += 1

            torch.cuda.synchronize()
            history.append({
                "epoch": epoch,
                "learning_rate": lr_used,
                "train_loss": loss_sum / count,
                "train_accuracy": correct / count,
                "val_loss": val_loss / len(truth),
                "val_accuracy": float(accuracy_score(truth, predictions)),
                "val_balanced_accuracy": float(
                    balanced_accuracy_score(truth, predictions)
                ),
                "val_macro_f1": score,
                "epoch_seconds": time.perf_counter() - started,
                "best_epoch_so_far": best_epoch,
            })
            scheduler.step()

            save_torch({
                "config": config, "epoch": epoch,
                "model": cpu_state(model),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "scaler": scaler.state_dict(),
                "history": history,
                "best_score": best_score, "best_epoch": best_epoch,
                "best_weights": best_weights, "bad_epochs": bad_epochs,
            }, latest)

            if improved:
                save_torch({
                    "config": config, "epoch": best_epoch,
                    "validation_macro_f1": best_score, "model": best_weights,
                }, folder / "best_model.pt")
            save_csv(history, folder / "training_log.csv")

            row = history[-1]
            print(
                f"{crop.upper()} | {epoch:02d}/{EPOCHS} | "
                f"val acc {row['val_accuracy']:.4f} | macro-F1 {score:.4f} | "
                f"{row['epoch_seconds']/60:.1f} min | "
                f"{'BEST' if improved else 'saved'}", flush=True
            )
            if bad_epochs >= PATIENCE:
                print("Early stopping.", flush=True)
                break

        del model, optimizer, scheduler, scaler, loaders
        del images, targets, logits, loss
        gc.collect()
        torch.cuda.empty_cache()

    if not history or best_weights is None:
        raise RuntimeError("No completed epoch available.")

    save_torch({
        "config": config, "epoch": best_epoch,
        "validation_macro_f1": best_score, "model": best_weights,
    }, folder / "best_model.pt")
    save_csv(history, folder / "training_log.csv")
    plot_history(history, folder)

    selected = next(row for row in history if row["epoch"] == best_epoch)
    summary = {
        "crop": crop, "model": "EfficientNet-B0",
        "experiment": "degradation", "seed": SEED,
        "best_epoch": best_epoch, "epochs_completed": len(history),
        "val_accuracy": selected["val_accuracy"],
        "val_balanced_accuracy": selected["val_balanced_accuracy"],
        "val_macro_f1": selected["val_macro_f1"],
        "test_evaluated": False,
    }
    save_json(summary, folder / "validation_summary.json")
    return summary


try:
    summaries = []
    for crop in ("wheat", "rice"):
        summaries.append(train_crop(crop))
        save_csv(summaries, ROOT / "metrics/degradation_validation_summary.csv")
        backup(f"{crop} degradation complete")

    comparison = []
    for crop in ("wheat", "rice"):
        for architecture, experiment in (
            ("efficientnet_b0", "baseline"),
            ("mobilenet_v2", "baseline"),
            ("efficientnet_b0", "degradation"),
        ):
            folder = ROOT / "runs" / f"{crop}_{architecture}_{experiment}_seed42"
            row = json.loads((folder / "validation_summary.json").read_text())
            row["experiment"] = experiment
            comparison.append(row)

    save_csv(
        comparison, ROOT / "metrics/all_experiments_validation_comparison.csv"
    )
    for path, expected in baseline_hashes.items():
        if digest(path) != expected:
            raise RuntimeError(f"Baseline checkpoint changed: {path}")

    print("\nALL SIX RUNS COMPLETE — VALIDATION RESULTS ONLY", flush=True)
    display(pd.DataFrame(comparison))
    backup("ALL SIX RUNS COMPLETE")
    print("CELL 2 COMPLETE. Test images were not evaluated.", flush=True)

except (Exception, KeyboardInterrupt):
    print("Interrupted. Attempting partial backup...", flush=True)
    try:
        backup("PARTIAL DEGRADATION RUN")
    except Exception as error:
        print("Partial backup failed:", error, flush=True)
    raise

In [ ]:
# Check saved files before final test evaluation — NO TRAINING
from pathlib import Path
import json
import pandas as pd

ROOT = Path("/kaggle/working/wheat_rice_journal")
experiments = [
    "efficientnet_b0_baseline",
    "mobilenet_v2_baseline",
    "efficientnet_b0_degradation",
]

print("SIX-MODEL CHECK")
missing = []

for crop in ["wheat", "rice"]:
    for experiment in experiments:
        run = ROOT / "runs" / f"{crop}_{experiment}_seed42"
        required = [
            run / "best_model.pt",
            run / "config.json",
            run / "validation_summary.json",
        ]
        absent = [p.name for p in required if not p.is_file()]
        print(f"{crop} / {experiment}:",
              "PRESENT" if not absent else f"MISSING {absent}")
        missing.extend(str(run / name) for name in absent)

for relative in [
    "splits/split_config.json",
    "data_audit/class_maps.json",
    "splits/wheat_test.csv",
    "splits/rice_test.csv",
]:
    if not (ROOT / relative).is_file():
        missing.append(str(ROOT / relative))

if missing:
    print("\nRestore the downloaded six-model backup before evaluation.")
    print("\n".join(missing))
else:
    print("\nSAVED MODEL CONFIGURATIONS")
    for experiment in experiments:
        path = ROOT / "runs" / f"wheat_{experiment}_seed42" / "config.json"
        print(f"\n{experiment}")
        print(json.dumps(json.loads(path.read_text()), indent=2))

    print("\nSPLIT CONFIGURATION")
    print((ROOT / "splits/split_config.json").read_text())

    print("\nCLASS MAPS")
    print((ROOT / "data_audit/class_maps.json").read_text())

    print("\nTEST FILE STRUCTURE")
    for crop in ["wheat", "rice"]:
        df = pd.read_csv(ROOT / "splits" / f"{crop}_test.csv")
        print(f"\n{crop}: {len(df)} test images")
        print("Columns:", df.columns.tolist())
        print(df.head(1).to_string(index=False))

    print("\nCHECK COMPLETE — no test predictions generated.")

In [2]:
# FINAL CLEAN TEST EVALUATION — ALL SIX MODELS
# No training. No pretrained-weight downloads.
# Uses best checkpoints selected on validation macro-F1.

from pathlib import Path
import os
import json
import hashlib
import time
import gc

import numpy as np
import pandas as pd
from PIL import Image, ImageOps

import torch
from torch import nn
from torchvision import models, transforms
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_recall_fscore_support,
    classification_report, confusion_matrix
)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path("/kaggle/working/wheat_rice_journal")
OUT = ROOT / "final_evaluation" / "clean"
OUT.mkdir(parents=True, exist_ok=True)

EXPERIMENTS = [
    "efficientnet_b0_baseline",
    "mobilenet_v2_baseline",
    "efficientnet_b0_degradation",
]
BATCH_SIZE = 32

if not torch.cuda.is_available():
    raise RuntimeError("GPU unavailable. Enable the T4 GPU before this cell.")

DEVICE = torch.device("cuda:0")
torch.set_num_threads(2)
torch.manual_seed(42)
np.random.seed(42)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

def read_json(path):
    return json.loads(Path(path).read_text())

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def save_json(data, path):
    path = Path(path)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(data, indent=2))
    os.replace(temporary, path)

def save_csv(df, path):
    path = Path(path)
    temporary = path.with_name(path.name + ".tmp")
    df.to_csv(temporary, index=False)
    os.replace(temporary, path)

split_config = read_json(ROOT / "splits/split_config.json")
class_maps = read_json(ROOT / "data_audit/class_maps.json")
split_hashes = split_config["split_sha256"]

print("Checking all six original split hashes...", flush=True)
for crop in ["wheat", "rice"]:
    for split in ["train", "validation", "test"]:
        name = f"{crop}_{split}.csv"
        path = ROOT / "splits" / name
        if not path.is_file() or sha256(path) != split_hashes[name]:
            raise RuntimeError(f"Split verification failed: {name}")

# Validate every run before beginning test inference.
run_info = {}
frames = {}

for crop, expected_count in [("wheat", 1634), ("rice", 1553)]:
    cmap = class_maps[crop]
    if sorted(cmap.values()) != list(range(len(cmap))):
        raise RuntimeError(f"{crop}: class IDs are not contiguous.")

    splits = {
        s: pd.read_csv(ROOT / "splits" / f"{crop}_{s}.csv")
        for s in ["train", "validation", "test"]
    }

    for a, b in [
        ("train", "validation"), ("train", "test"),
        ("validation", "test")
    ]:
        for key in ["group_id", "pixel_sha256"]:
            if set(splits[a][key]) & set(splits[b][key]):
                raise RuntimeError(f"{crop}: {key} overlap: {a}/{b}")

    df = splits["test"]
    if len(df) != expected_count:
        raise RuntimeError(f"{crop}: unexpected test image count.")
    if df["path"].duplicated().any():
        raise RuntimeError(f"{crop}: repeated test paths.")
    if not df["label"].map(cmap).eq(df["label_id"]).all():
        raise RuntimeError(f"{crop}: label mapping mismatch.")
    if set(df["label_id"]) != set(cmap.values()):
        raise RuntimeError(f"{crop}: some classes missing from test.")

    missing = [p for p in df["path"] if not Path(p).is_file()]
    if missing:
        raise FileNotFoundError(
            f"{crop}: {len(missing)} image paths missing. "
            f"First missing path: {missing[0]}"
        )
    frames[crop] = df

    for experiment in EXPERIMENTS:
        run = ROOT / "runs" / f"{crop}_{experiment}_seed42"
        cfg = read_json(run / "config.json")
        summary = read_json(run / "validation_summary.json")
        checkpoint = run / "best_model.pt"

        if cfg["class_map"] != cmap:
            raise RuntimeError(f"Class map mismatch: {run.name}")
        if cfg["split_sha256"] != split_hashes:
            raise RuntimeError(f"Split configuration mismatch: {run.name}")
        if not checkpoint.is_file():
            raise FileNotFoundError(checkpoint)

        run_info[(crop, experiment)] = (cfg, summary, checkpoint)

resize_crop = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
])
normalize = transforms.Normalize(
    [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
)

protocol = {
    "condition": "clean",
    "checkpoint_selection": "best clean validation macro-F1",
    "preprocessing": "EXIF transpose; RGB; bilinear Resize(256); "
                     "CenterCrop(224); ImageNet normalization",
    "precision": "float32",
    "batch_size": BATCH_SIZE,
    "split_sha256": split_hashes,
    "torch_version": str(torch.__version__),
    "gpu": torch.cuda.get_device_name(0),
}
save_json(protocol, OUT / "evaluation_protocol.json")

print("Checks passed. GPU:", torch.cuda.get_device_name(0), flush=True)
rows = []

for crop in ["wheat", "rice"]:
    df = frames[crop]
    cmap = class_maps[crop]
    names = sorted(cmap, key=cmap.get)
    labels = list(range(len(names)))
    y_true = df["label_id"].to_numpy(dtype=np.int64)

    # Decode once per crop; reuse identical images for all three models.
    print(f"\n{crop.upper()}: preparing {len(df):,} test images...",
          flush=True)
    images = np.empty((len(df), 224, 224, 3), dtype=np.uint8)

    for i, path in enumerate(df["path"]):
        with Image.open(path) as im:
            im = ImageOps.exif_transpose(im).convert("RGB")
            images[i] = np.asarray(resize_crop(im))
        if (i + 1) % 400 == 0 or i + 1 == len(df):
            print(f"  Prepared {i + 1}/{len(df)}", flush=True)

    for experiment in EXPERIMENTS:
        started = time.time()
        cfg, summary, checkpoint_path = run_info[(crop, experiment)]
        destination = OUT / f"{crop}_{experiment}_seed42"
        destination.mkdir(parents=True, exist_ok=True)

        fingerprint = {
            "checkpoint_sha256": sha256(checkpoint_path),
            "protocol": protocol,
            "crop": crop,
            "class_map": cmap,
        }
        cache_path = destination / "probabilities.npy"
        metadata_path = destination / "prediction_metadata.json"

        if (
            cache_path.is_file()
            and metadata_path.is_file()
            and read_json(metadata_path) == fingerprint
        ):
            probabilities = np.load(cache_path, allow_pickle=False)
            print(f"\n{crop} / {experiment}: loading verified cache",
                  flush=True)
        else:
            print(f"\n{crop} / {experiment}: evaluating...", flush=True)
            if cfg["architecture"] == "torchvision_efficientnet_b0":
                model = models.efficientnet_b0(weights=None)
            elif cfg["architecture"] == "torchvision_mobilenet_v2":
                model = models.mobilenet_v2(weights=None)
            else:
                raise RuntimeError(f"Unsupported model: {cfg['architecture']}")

            model.classifier[1] = nn.Linear(
                model.classifier[1].in_features, len(names)
            )

            # Load only your own saved checkpoint.
            checkpoint = torch.load(
                checkpoint_path, map_location="cpu", weights_only=False
            )
            if int(checkpoint["epoch"]) != int(summary["best_epoch"]):
                raise RuntimeError("Checkpoint epoch does not match summary.")

            model.load_state_dict(checkpoint["model"], strict=True)
            del checkpoint
            model = model.to(DEVICE).eval()

            batches = []
            with torch.inference_mode():
                for start in range(0, len(df), BATCH_SIZE):
                    x = torch.from_numpy(
                        images[start:start + BATCH_SIZE]
                    ).permute(0, 3, 1, 2).to(
                        DEVICE, dtype=torch.float32
                    ).div_(255.0)
                    x = normalize(x)
                    batches.append(
                        model(x).softmax(dim=1).cpu().numpy()
                    )
                    batch_number = start // BATCH_SIZE + 1
                    if batch_number == 1 or batch_number % 20 == 0:
                        print(f"  Batch {batch_number}/"
                              f"{(len(df) + BATCH_SIZE - 1)//BATCH_SIZE}",
                              flush=True)

            probabilities = np.concatenate(batches)
            temp_cache = destination / "probabilities.tmp"
            with open(temp_cache, "wb") as f:
                np.save(f, probabilities, allow_pickle=False)
            os.replace(temp_cache, cache_path)
            save_json(fingerprint, metadata_path)

            del model, x, batches
            gc.collect()
            torch.cuda.empty_cache()

        if probabilities.shape != (len(df), len(names)):
            raise RuntimeError("Prediction shape mismatch.")
        if not np.isfinite(probabilities).all():
            raise RuntimeError("Non-finite prediction probabilities.")
        if not np.allclose(probabilities.sum(axis=1), 1, atol=1e-5):
            raise RuntimeError("Invalid probability sums.")

        y_pred = probabilities.argmax(axis=1)
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, labels=labels,
            average="macro", zero_division=0
        )
        row = {
            "crop": crop,
            "experiment": experiment,
            "seed": 42,
            "selected_epoch": int(summary["best_epoch"]),
            "test_images": len(df),
            "correct": int((y_true == y_pred).sum()),
            "accuracy": float(accuracy_score(y_true, y_pred)),
            "balanced_accuracy": float(
                balanced_accuracy_score(y_true, y_pred)
            ),
            "macro_precision": float(precision),
            "macro_recall": float(recall),
            "macro_f1": float(f1),
            "condition": "clean",
        }
        rows.append(row)
        save_json(row, destination / "test_metrics.json")

        predictions = df[
            ["image_id", "path", "group_id", "label", "label_id"]
        ].copy()
        predictions["predicted_id"] = y_pred
        predictions["predicted_label"] = [names[i] for i in y_pred]
        predictions["confidence"] = probabilities.max(axis=1)
        predictions["correct"] = y_true == y_pred
        for j, name in enumerate(names):
            predictions[f"prob_{name}"] = probabilities[:, j]
        save_csv(predictions, destination / "test_predictions.csv")

        report = classification_report(
            y_true, y_pred, labels=labels, target_names=names,
            output_dict=True, zero_division=0
        )
        per_class = pd.DataFrame(
            [{"class": name, **report[name]} for name in names]
        )
        save_csv(per_class, destination / "per_class_metrics.csv")

        cm = confusion_matrix(y_true, y_pred, labels=labels)
        pd.DataFrame(cm, index=names, columns=names).to_csv(
            destination / "confusion_matrix_counts.csv",
            index_label="true_class"
        )

        fig, ax = plt.subplots(figsize=(12, 10))
        heatmap = ax.imshow(cm, cmap="Blues")
        fig.colorbar(heatmap, ax=ax, fraction=0.046, pad=0.04)
        ax.set(
            xticks=labels, yticks=labels,
            xticklabels=names, yticklabels=names,
            xlabel="Predicted class", ylabel="True class",
            title=f"{crop.title()} — {experiment}\nClean held-out test"
        )
        plt.setp(ax.get_xticklabels(), rotation=60, ha="right")
        for i in labels:
            for j in labels:
                ax.text(
                    j, i, str(cm[i, j]), ha="center", va="center",
                    fontsize=7,
                    color="white" if cm[i, j] > cm.max()/2 else "black"
                )
        fig.tight_layout()
        fig.savefig(destination / "confusion_matrix.png", dpi=300)
        fig.savefig(destination / "confusion_matrix.pdf")
        plt.close(fig)

        # Save progress after each completed model.
        save_csv(pd.DataFrame(rows), OUT / "clean_test_comparison.csv")
        print(
            f"  Accuracy: {row['accuracy']:.4f} | "
            f"Macro-F1: {row['macro_f1']:.4f} | "
            f"{(time.time()-started)/60:.1f} min | SAVED",
            flush=True
        )

    del images
    gc.collect()

results = pd.DataFrame(rows)
paper_table = results[
    ["crop", "experiment", "selected_epoch", "test_images",
     "accuracy", "balanced_accuracy", "macro_f1"]
].copy()

for column in ["accuracy", "balanced_accuracy", "macro_f1"]:
    paper_table[column] = (paper_table[column] * 100).round(2)

paper_table.rename(columns={
    "accuracy": "accuracy_percent",
    "balanced_accuracy": "balanced_accuracy_percent",
    "macro_f1": "macro_f1_percent"
}, inplace=True)
save_csv(paper_table, OUT / "clean_test_table_for_word.csv")

print("\nCLEAN TEST EVALUATION COMPLETE — ALL SIX MODELS")
display(paper_table)
print("\nSaved to:", OUT)
print("Models and original splits were not changed.")
print("Next: fixed degradation evaluation. No further model tuning.")

Checking all six original split hashes...
Checks passed. GPU: Tesla T4

WHEAT: preparing 1,634 test images...
  Prepared 400/1634
  Prepared 800/1634
  Prepared 1200/1634
  Prepared 1600/1634
  Prepared 1634/1634

wheat / efficientnet_b0_baseline: evaluating...
  Batch 1/52
  Batch 20/52
  Batch 40/52
  Accuracy: 0.8482 | Macro-F1: 0.8288 | 0.3 min | SAVED

wheat / mobilenet_v2_baseline: evaluating...
  Batch 1/52
  Batch 20/52
  Batch 40/52
  Accuracy: 0.8433 | Macro-F1: 0.8208 | 0.1 min | SAVED

wheat / efficientnet_b0_degradation: evaluating...
  Batch 1/52
  Batch 20/52
  Batch 40/52
  Accuracy: 0.8599 | Macro-F1: 0.8384 | 0.1 min | SAVED

RICE: preparing 1,553 test images...
  Prepared 400/1553
  Prepared 800/1553
  Prepared 1200/1553
  Prepared 1553/1553

rice / efficientnet_b0_baseline: evaluating...
  Batch 1/49
  Batch 20/49
  Batch 40/49
  Accuracy: 0.9562 | Macro-F1: 0.9484 | 0.1 min | SAVED

rice / mobilenet_v2_baseline: evaluating...
  Batch 1/49
  Batch 20/49
  Batch 40/4

,crop,experiment,selected_epoch,test_images,accuracy_percent,balanced_accuracy_percent,macro_f1_percent
0,wheat,efficientnet_b0_baseline,13,1634,84.82,82.48,82.88
1,wheat,mobilenet_v2_baseline,13,1634,84.33,82.03,82.08
2,wheat,efficientnet_b0_degradation,13,1634,85.99,83.52,83.84
3,rice,efficientnet_b0_baseline,15,1553,95.62,95.20,94.84
4,rice,mobilenet_v2_baseline,13,1553,95.30,95.34,94.95
5,rice,efficientnet_b0_degradation,10,1553,95.62,95.69,95.50



Saved to: /kaggle/working/wheat_rice_journal/final_evaluation/clean
Models and original splits were not changed.
Next: fixed degradation evaluation. No further model tuning.


In [ ]:
from pathlib import Path
import zipfile
import hashlib
import json
from IPython.display import display, FileLink

root = Path("/kaggle/working/wheat_rice_journal")
source = root / "final_evaluation"
destination = Path("/kaggle/working/wheat_rice_test_results_backup.zip")

required = source / "clean/clean_test_comparison.csv"
if not required.is_file():
    raise FileNotFoundError("Clean test results missing; backup not created.")

files = sorted(p for p in source.rglob("*") if p.is_file())
hashes = {}

print("Backing up test results...", flush=True)
with zipfile.ZipFile(
    destination, "w", compression=zipfile.ZIP_DEFLATED
) as archive:
    for path in files:
        name = path.relative_to(root).as_posix()
        data = path.read_bytes()
        hashes[name] = hashlib.sha256(data).hexdigest()
        archive.writestr(name, data)
    archive.writestr(
        "RESULTS_MANIFEST.json", json.dumps(hashes, indent=2)
    )

with zipfile.ZipFile(destination) as archive:
    for name, expected in hashes.items():
        if hashlib.sha256(archive.read(name)).hexdigest() != expected:
            raise RuntimeError(f"Backup verification failed: {name}")

print(f"BACKUP VERIFIED — {len(files)} files")
display(FileLink(str(destination), result_html_prefix="Download: "))

In [ ]:
from pathlib import Path
from IPython.display import display, HTML
import shutil
import zipfile

original = Path("/kaggle/working/wheat_rice_test_results_backup.zip")
download = Path("/kaggle/working/test_results.zip")

if not original.is_file():
    raise FileNotFoundError(
        "Backup ZIP missing. Pichhla backup banane wala cell run karein."
    )

with zipfile.ZipFile(original) as archive:
    bad_file = archive.testzip()
    if bad_file:
        raise RuntimeError(f"ZIP verification failed: {bad_file}")

shutil.copy2(original, download)

print(f"ZIP ready: {download.stat().st_size / (1024**2):.2f} MB")
display(HTML(
    '<a href="files/test_results.zip" download="test_results.zip" '
    'target="_blank"><b>CLICK HERE — Download test results</b></a>'
))

In [1]:
from pathlib import Path

for root in [Path("/kaggle/input"), Path("/kaggle/working")]:
    print(f"\nLOCATION: {root}")

    if not root.exists():
        print("Folder not found.")
        continue

    matches = sorted(
        p for p in root.rglob("*")
        if p.is_file()
        and p.suffix.lower() in {
            ".pt", ".pth", ".ckpt", ".csv", ".json", ".zip"
        }
    )

    print(f"Matching files: {len(matches)}")

    for p in matches[:150]:
        size_mb = p.stat().st_size / (1024 * 1024)
        print(f"{size_mb:8.2f} MB | {p}")

    if len(matches) > 150:
        print(f"... {len(matches) - 150} more files.")


LOCATION: /kaggle/input
Matching files: 58
    0.04 MB | /kaggle/input/competitions/paddy-disease-classification/sample_submission.csv
    0.32 MB | /kaggle/input/competitions/paddy-disease-classification/train.csv
    0.01 MB | /kaggle/input/datasets/dollynijhawancu/wheat-rice-latest-verified-latest/BACKUP_MANIFEST.json
    0.00 MB | /kaggle/input/datasets/dollynijhawancu/wheat-rice-latest-verified-latest/wheat_rice_journal/data_audit/class_maps.json
    0.00 MB | /kaggle/input/datasets/dollynijhawancu/wheat-rice-latest-verified-latest/wheat_rice_journal/data_audit/cleaning_split_summary.csv
    0.15 MB | /kaggle/input/datasets/dollynijhawancu/wheat-rice-latest-verified-latest/wheat_rice_journal/data_audit/exact_duplicate_label_conflicts.csv
    1.20 MB | /kaggle/input/datasets/dollynijhawancu/wheat-rice-latest-verified-latest/wheat_rice_journal/data_audit/exact_duplicate_members.csv
    0.66 MB | /kaggle/input/datasets/dollynijhawancu/wheat-rice-latest-verified-latest/wheat_rice_jou

In [2]:
# CHECK SAVED CONFIGURATION AND SPLITS — NO TRAINING

from pathlib import Path
import json
import pandas as pd

BASE = Path(
    "/kaggle/input/datasets/dollynijhawancu/"
    "wheat-rice-latest-verified-latest/wheat_rice_journal"
)

assert BASE.is_dir(), f"Backup folder not found: {BASE}"


def show_json(path):
    print(f"\n--- {path.relative_to(BASE)} ---")

    if not path.exists():
        print("MISSING")
        return

    try:
        content = json.loads(path.read_text())
        print(json.dumps(content, indent=2))
    except Exception as error:
        print("Could not read:", error)


# 1. Class labels and original split protocol
show_json(BASE / "data_audit/class_maps.json")
show_json(BASE / "splits/split_config.json")

# 2. Dataset-cleaning summary
summary_path = BASE / "data_audit/cleaning_split_summary.csv"

if summary_path.exists():
    print("\n--- CLEANING AND SPLIT SUMMARY ---")
    print(pd.read_csv(summary_path).to_string(index=False))

# 3. Actual saved model configuration
for run_dir in sorted((BASE / "runs").iterdir()):
    if not run_dir.is_dir():
        continue

    print(f"\n========== {run_dir.name} ==========")

    for filename in ["best_model.pt", "latest_checkpoint.pt"]:
        path = run_dir / filename
        if path.exists():
            print(
                f"{filename}: PRESENT "
                f"({path.stat().st_size / 1024**2:.2f} MB)"
            )
        else:
            print(f"{filename}: MISSING")

    show_json(run_dir / "config.json")
    show_json(run_dir / "validation_summary.json")

# 4. Inspect all six split CSVs
# Print headers and one example so path/label names are not guessed.
for crop in ["wheat", "rice"]:
    split_frames = {}

    for split in ["train", "validation", "test"]:
        path = BASE / "splits" / f"{crop}_{split}.csv"

        print(f"\n--- {crop.upper()} / {split.upper()} ---")

        if not path.exists():
            print("MISSING")
            continue

        frame = pd.read_csv(path)
        split_frames[split] = frame

        print("Rows:", len(frame))
        print("Columns:", frame.columns.tolist())
        print("First row:")
        print(frame.head(1).to_string(index=False))

    # Check shared group/hash identifiers across partitions, if present.
    if len(split_frames) == 3:
        common_columns = set.intersection(
            *(set(frame.columns) for frame in split_frames.values())
        )

        identifier_columns = sorted(
            column for column in common_columns
            if any(
                term in column.lower()
                for term in ["group", "hash", "sha256", "md5"]
            )
        )

        print(f"\n--- {crop.upper()}: CROSS-SPLIT IDENTIFIER CHECK ---")

        if not identifier_columns:
            print("No group/hash columns found; further audit needed.")

        for column in identifier_columns:
            values = {
                split: set(frame[column].dropna().astype(str))
                for split, frame in split_frames.items()
            }

            for first, second in [
                ("train", "validation"),
                ("train", "test"),
                ("validation", "test"),
            ]:
                overlap = values[first] & values[second]

                print(
                    f"{column}: {first} vs {second} "
                    f"= {len(overlap)} shared values"
                )

print("\nREAD-ONLY CHECK COMPLETE — no files changed.")


--- data_audit/class_maps.json ---
{
  "rice": {
    "bacterial_leaf_blight": 0,
    "bacterial_leaf_streak": 1,
    "bacterial_panicle_blight": 2,
    "blast": 3,
    "brown_spot": 4,
    "dead_heart": 5,
    "downy_mildew": 6,
    "hispa": 7,
    "normal": 8,
    "tungro": 9
  },
  "wheat": {
    "aphid": 0,
    "black_rust": 1,
    "blast": 2,
    "brown_rust": 3,
    "common_root_rot": 4,
    "fusarium_head_blight": 5,
    "healthy": 6,
    "leaf_blight": 7,
    "mildew": 8,
    "mite": 9,
    "septoria": 10,
    "smut": 11,
    "stem_fly": 12,
    "tan_spot": 13,
    "yellow_rust": 14
  }
}

--- splits/split_config.json ---
{
  "audit_sha256": "db6be37703f3a25b9a55a6ede0c031cbe111971eb2bd3b7efeefcb576d49c4a5",
  "seed": 42,
  "phash_distance": 4,
  "target_fractions": {
    "train": 0.7,
    "validation": 0.15,
    "test": 0.15
  },
  "protocol": "New grouped experimental splits; original splits replaced",
  "split_sha256": {
    "rice_train.csv": "f357f379745ca0c69bee60dd761c8f7

In [3]:
# RECOVERY CHECK:
# Verify saved split hashes, load trained models,
# reproduce validation scores and save validation predictions.
# NO TRAINING. NO TEST EVALUATION.

from pathlib import Path
from collections.abc import Mapping
import json
import hashlib
import os
import gc
import time

import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import accuracy_score, f1_score
from IPython.display import display


BASE = Path(
    "/kaggle/input/datasets/dollynijhawancu/"
    "wheat-rice-latest-verified-latest/wheat_rice_journal"
)

# Separate output directory; original backup remains unchanged.
OUTPUT = Path("/kaggle/working/wheat_rice_recovery")
OUTPUT.mkdir(parents=True, exist_ok=True)

assert BASE.is_dir(), f"Backup missing: {BASE}"
assert torch.cuda.is_available(), "Enable GPU, then run this cell."

DEVICE = torch.device("cuda:0")

print("GPU:", torch.cuda.get_device_name(0))


def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def save_json(content, path):
    temporary = Path(str(path) + ".tmp")
    temporary.write_text(
        json.dumps(content, indent=2, default=str)
    )
    os.replace(temporary, path)


# 1. Verify all original split files before using them.
split_config = json.loads(
    (BASE / "splits/split_config.json").read_text()
)

expected_hashes = split_config["split_sha256"]

assert len(expected_hashes) == 6, (
    "Expected six original split hashes."
)

for filename, expected in expected_hashes.items():
    path = BASE / "splits" / filename

    assert path.is_file(), f"Missing split: {path}"
    actual = file_sha256(path)

    assert actual == expected, (
        f"Split hash mismatch: {filename}. "
        "Stopped without changing any split."
    )

print("All six split-file hashes match the saved protocol.")

class_maps = json.loads(
    (BASE / "data_audit/class_maps.json").read_text()
)


# 2. Reconstruct the saved validation transform.
evaluation_transform = transforms.Compose([
    transforms.Resize(
        256,
        interpolation=transforms.InterpolationMode.BILINEAR,
        antialias=True,
    ),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])


class ValidationDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]

        with Image.open(row["path"]) as image:
            image = image.convert("RGB")
            tensor = evaluation_transform(image)

        return tensor, int(row["label_id"])


def extract_state_dict(checkpoint):
    # Supports common saved-checkpoint formats.
    if isinstance(checkpoint, Mapping):
        if checkpoint and all(
            torch.is_tensor(value)
            for value in checkpoint.values()
        ):
            state = checkpoint
        else:
            state = None

            for key in [
                "model_state_dict",
                "state_dict",
                "model",
                "model_state",
            ]:
                candidate = checkpoint.get(key)

                if (
                    isinstance(candidate, Mapping)
                    and candidate
                    and all(
                        torch.is_tensor(value)
                        for value in candidate.values()
                    )
                ):
                    state = candidate
                    break

            if state is None:
                raise RuntimeError(
                    "Unrecognised checkpoint structure. "
                    f"Top-level keys: {list(checkpoint.keys())}"
                )
    else:
        raise RuntimeError(
            f"Unexpected checkpoint type: {type(checkpoint)}"
        )

    # Handle DataParallel checkpoints if present.
    if all(key.startswith("module.") for key in state):
        state = {
            key.removeprefix("module."): value
            for key, value in state.items()
        }

    return state


def create_model(architecture, number_of_classes):
    # No pretrained download: saved trained weights will be loaded.
    if architecture == "torchvision_efficientnet_b0":
        model = models.efficientnet_b0(weights=None)
        model.classifier[1] = nn.Linear(
            model.classifier[1].in_features,
            number_of_classes,
        )

    elif architecture == "torchvision_mobilenet_v2":
        model = models.mobilenet_v2(weights=None)
        model.classifier[1] = nn.Linear(
            model.classifier[1].in_features,
            number_of_classes,
        )

    else:
        raise RuntimeError(
            f"Unsupported saved architecture: {architecture}"
        )

    return model


# 3. Validate paths and class IDs before GPU inference.
frames = {}

for crop in ["wheat", "rice"]:
    frame = pd.read_csv(
        BASE / "splits" / f"{crop}_validation.csv"
    )

    required = {"path", "label", "label_id", "pixel_sha256"}
    assert required.issubset(frame.columns)

    assert not frame[list(required)].isna().any().any(), (
        f"{crop}: missing path, label or identifier."
    )

    missing = [
        path for path in frame["path"]
        if not Path(path).is_file()
    ]

    assert not missing, (
        f"{crop}: {len(missing)} image paths unavailable. "
        f"Examples: {missing[:3]}"
    )

    expected_ids = frame["label"].map(class_maps[crop])

    assert expected_ids.notna().all()
    assert np.array_equal(
        expected_ids.to_numpy(dtype=int),
        frame["label_id"].to_numpy(dtype=int),
    ), f"{crop}: class mapping mismatch."

    frames[crop] = frame.reset_index(drop=True)
    print(f"{crop}: {len(frame)} validation paths and labels checked.")


# 4. Reproduce validation scores for all four models.
rows = []

for crop in ["wheat", "rice"]:
    frame = frames[crop]

    data_loader = DataLoader(
        ValidationDataset(frame),
        batch_size=32,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
    )

    for model_name in [
        "efficientnet_b0_baseline",
        "mobilenet_v2_baseline",
    ]:
        run_name = f"{crop}_{model_name}_seed42"
        run = BASE / "runs" / run_name
        destination = OUTPUT / run_name
        destination.mkdir(exist_ok=True)

        config_path = run / "config.json"
        checkpoint_path = run / "best_model.pt"

        config = json.loads(config_path.read_text())
        saved_summary = json.loads(
            (run / "validation_summary.json").read_text()
        )

        assert config["crop"] == crop
        assert config["class_map"] == class_maps[crop]
        assert config["split_sha256"] == expected_hashes

        assert str(evaluation_transform) == config["eval_transform"], (
            f"{run_name}: transform representation differs. "
            "Stop and share this message so we can check compatibility."
        )

        # Cache is tied to checkpoint, config and validation split.
        fingerprint = hashlib.sha256(
            (
                file_sha256(checkpoint_path)
                + file_sha256(config_path)
                + expected_hashes[f"{crop}_validation.csv"]
                + "validation_recovery_v1"
            ).encode()
        ).hexdigest()

        cache_path = destination / "validation_logits.npz"
        metadata_path = destination / "validation_cache.json"

        cache_valid = (
            cache_path.exists()
            and metadata_path.exists()
            and json.loads(metadata_path.read_text()).get("fingerprint")
            == fingerprint
        )

        if cache_valid:
            with np.load(cache_path, allow_pickle=False) as saved:
                logits = saved["logits"]
                labels = saved["labels"]

            print(f"\n{run_name}: using matching saved predictions.")

        else:
            print(f"\n{run_name}: evaluating validation images...")
            started = time.time()

            # These are your own trusted training checkpoints.
            checkpoint = torch.load(
                checkpoint_path,
                map_location="cpu",
                weights_only=False,
            )

            state = extract_state_dict(checkpoint)

            model = create_model(
                config["architecture"],
                len(class_maps[crop]),
            )

            model.load_state_dict(state, strict=True)
            model = model.to(DEVICE).eval()

            del checkpoint, state

            all_logits = []
            all_labels = []

            with torch.inference_mode():
                for batch_number, (images, targets) in enumerate(
                    data_loader, start=1
                ):
                    scores = model(
                        images.to(DEVICE, non_blocking=True)
                    )

                    all_logits.append(
                        scores.float().cpu().numpy()
                    )
                    all_labels.append(targets.numpy())

                    if batch_number == 1 or batch_number % 20 == 0:
                        print(
                            f"  Batch {batch_number}/{len(data_loader)}",
                            flush=True,
                        )

            logits = np.concatenate(all_logits)
            labels = np.concatenate(all_labels)

            temporary = destination / "validation_logits.tmp"

            with open(temporary, "wb") as file:
                np.savez_compressed(
                    file,
                    logits=logits,
                    labels=labels,
                )

            os.replace(temporary, cache_path)

            save_json(
                {
                    "fingerprint": fingerprint,
                    "seconds": time.time() - started,
                    "torch_version": torch.__version__,
                },
                metadata_path,
            )

            del model
            gc.collect()
            torch.cuda.empty_cache()

        assert np.isfinite(logits).all()
        assert logits.shape == (len(frame), len(class_maps[crop]))
        assert np.array_equal(
            labels,
            frame["label_id"].to_numpy(dtype=int),
        )

        predictions = logits.argmax(axis=1)

        accuracy = accuracy_score(labels, predictions)
        macro_f1 = f1_score(
            labels,
            predictions,
            labels=np.arange(len(class_maps[crop])),
            average="macro",
            zero_division=0,
        )

        saved_accuracy = saved_summary["val_accuracy"]
        saved_f1 = saved_summary["val_macro_f1"]

        # Small tolerance allows minor numerical differences.
        matches = (
            abs(accuracy - saved_accuracy) <= 0.001
            and abs(macro_f1 - saved_f1) <= 0.001
        )

        rows.append({
            "crop": crop,
            "model": model_name,
            "saved_val_accuracy": saved_accuracy,
            "reproduced_val_accuracy": accuracy,
            "saved_val_macro_f1": saved_f1,
            "reproduced_val_macro_f1": macro_f1,
            "matches_within_0.001": matches,
        })

        prediction_table = frame[
            ["path", "label", "label_id", "pixel_sha256"]
        ].copy()

        prediction_table["predicted_id"] = predictions
        prediction_table.to_csv(
            destination / "validation_predictions.csv",
            index=False,
        )

        # Incrementally preserve completed checks.
        pd.DataFrame(rows).to_csv(
            OUTPUT / "validation_reproduction.csv",
            index=False,
        )

        print(
            f"Accuracy: {accuracy:.4f} | "
            f"Macro-F1: {macro_f1:.4f} | "
            f"Matches saved: {matches}"
        )


comparison = pd.DataFrame(rows)

print("\nVALIDATION REPRODUCTION SUMMARY")
display(comparison)

if comparison["matches_within_0.001"].all():
    print("\nPASS: All four validation results reproduced within tolerance.")
else:
    print("\nCHECK NEEDED: Some results differ. Share the table before proceeding.")

print("Saved outputs:", OUTPUT)
print("No training performed. No test evaluation performed.")

GPU: Tesla T4
All six split-file hashes match the saved protocol.
wheat: 1637 validation paths and labels checked.
rice: 1541 validation paths and labels checked.

wheat_efficientnet_b0_baseline_seed42: evaluating validation images...
  Batch 1/52
  Batch 20/52
  Batch 40/52
Accuracy: 0.8613 | Macro-F1: 0.8447 | Matches saved: False

wheat_mobilenet_v2_baseline_seed42: evaluating validation images...
  Batch 1/52
  Batch 20/52
  Batch 40/52
Accuracy: 0.8412 | Macro-F1: 0.8229 | Matches saved: False

rice_efficientnet_b0_baseline_seed42: evaluating validation images...
  Batch 1/49
  Batch 20/49
  Batch 40/49
Accuracy: 0.9650 | Macro-F1: 0.9662 | Matches saved: True

rice_mobilenet_v2_baseline_seed42: evaluating validation images...
  Batch 1/49
  Batch 20/49
  Batch 40/49
Accuracy: 0.9552 | Macro-F1: 0.9546 | Matches saved: True

VALIDATION REPRODUCTION SUMMARY


,crop,model,saved_val_accuracy,reproduced_val_accuracy,saved_val_macro_f1,reproduced_val_macro_f1,matches_within_0.001
0,wheat,efficientnet_b0_baseline,0.863164,0.861332,0.846885,0.844672,False
1,wheat,mobilenet_v2_baseline,0.844227,0.841173,0.827355,0.822870,False
2,rice,efficientnet_b0_baseline,0.964958,0.964958,0.966192,0.966192,True
3,rice,mobilenet_v2_baseline,0.955224,0.955224,0.954605,0.954605,True



CHECK NEEDED: Some results differ. Share the table before proceeding.
Saved outputs: /kaggle/working/wheat_rice_recovery
No training performed. No test evaluation performed.


In [4]:
# TARGETED DIAGNOSTIC — EXIF ORIENTATION
# No training. No test-set evaluation.
# Run in the same session as the previous recovery cell.

from PIL import Image, ImageOps
import numpy as np
import pandas as pd
import torch
import json
import gc

required_names = [
    "BASE", "OUTPUT", "DEVICE", "class_maps",
    "evaluation_transform", "create_model", "extract_state_dict"
]

missing = [name for name in required_names if name not in globals()]
assert not missing, (
    f"Previous recovery-cell definitions are missing: {missing}. "
    "Share this message before restarting anything."
)

frame = pd.read_csv(
    BASE / "splits/wheat_validation.csv"
).reset_index(drop=True)

diagnostic_dir = OUTPUT / "exif_diagnostic"
diagnostic_dir.mkdir(exist_ok=True)

orientation_rows = []
scan_errors = []

print("Checking wheat validation image orientation...")

for index, row in frame.iterrows():
    try:
        with Image.open(row["path"]) as image:
            orientation = image.getexif().get(274, 1)

        # EXIF values 2–8 describe a flip and/or rotation.
        if orientation in {2, 3, 4, 5, 6, 7, 8}:
            orientation_rows.append({
                "row_index": index,
                "path": row["path"],
                "orientation": orientation,
                "label_id": int(row["label_id"]),
            })

    except Exception as error:
        scan_errors.append({
            "path": row["path"],
            "error": str(error),
        })

pd.DataFrame(
    scan_errors, columns=["path", "error"]
).to_csv(
    diagnostic_dir / "scan_errors.csv", index=False
)

assert not scan_errors, (
    "Some images could not be checked. Share scan_errors.csv."
)

affected = pd.DataFrame(
    orientation_rows,
    columns=["row_index", "path", "orientation", "label_id"]
)

affected.to_csv(
    diagnostic_dir / "orientation_images.csv",
    index=False,
)

print("Images requiring an EXIF transform:", len(affected))

if affected.empty:
    print(
        "EXIF rotation/flip does not explain the mismatch. "
        "Next we need the original notebook's image-loading code."
    )

else:
    display(affected[["row_index", "orientation", "label_id"]])

    labels = frame["label_id"].to_numpy(dtype=int)
    comparison_rows = []

    for model_name in [
        "efficientnet_b0_baseline",
        "mobilenet_v2_baseline",
    ]:
        run_name = f"wheat_{model_name}_seed42"
        run_dir = BASE / "runs" / run_name

        config = json.loads(
            (run_dir / "config.json").read_text()
        )
        saved_summary = json.loads(
            (run_dir / "validation_summary.json").read_text()
        )

        cache_path = (
            OUTPUT / run_name / "validation_logits.npz"
        )
        assert cache_path.exists(), f"Missing cache: {cache_path}"

        with np.load(cache_path, allow_pickle=False) as cache:
            original_logits = cache["logits"].copy()
            cached_labels = cache["labels"].copy()

        assert np.array_equal(cached_labels, labels)
        assert original_logits.shape == (len(frame), 15)

        # Keep the original cached logits unchanged.
        candidate_logits = original_logits.copy()

        # Load only your own trusted checkpoint.
        checkpoint = torch.load(
            run_dir / "best_model.pt",
            map_location="cpu",
            weights_only=False,
        )

        model = create_model(config["architecture"], 15)
        model.load_state_dict(
            extract_state_dict(checkpoint),
            strict=True,
        )
        model = model.to(DEVICE).eval()
        del checkpoint

        print(f"\nChecking orientation correction: {run_name}")

        with torch.inference_mode():
            for start in range(0, len(affected), 32):
                batch = affected.iloc[start:start + 32]
                tensors = []

                for path in batch["path"]:
                    with Image.open(path) as image:
                        image = ImageOps.exif_transpose(image)
                        image = image.convert("RGB")
                        tensors.append(evaluation_transform(image))

                images = torch.stack(tensors).to(DEVICE)
                logits = model(images).float().cpu().numpy()

                indices = batch["row_index"].to_numpy(dtype=int)
                candidate_logits[indices] = logits

        from sklearn.metrics import accuracy_score, f1_score

        original_predictions = original_logits.argmax(1)
        candidate_predictions = candidate_logits.argmax(1)

        accuracy = accuracy_score(labels, candidate_predictions)
        macro_f1 = f1_score(
            labels,
            candidate_predictions,
            labels=np.arange(15),
            average="macro",
            zero_division=0,
        )

        comparison_rows.append({
            "model": model_name,
            "orientation_images": len(affected),
            "changed_predictions": int(
                (original_predictions != candidate_predictions).sum()
            ),
            "saved_accuracy": saved_summary["val_accuracy"],
            "without_exif_accuracy": accuracy_score(
                labels, original_predictions
            ),
            "with_exif_accuracy": accuracy,
            "saved_macro_f1": saved_summary["val_macro_f1"],
            "with_exif_macro_f1": macro_f1,
            "matches_saved_metrics": bool(
                abs(accuracy - saved_summary["val_accuracy"]) < 1e-8
                and abs(macro_f1 - saved_summary["val_macro_f1"]) < 1e-8
            ),
        })

        np.savez_compressed(
            diagnostic_dir / f"{run_name}_exif_candidate.npz",
            logits=candidate_logits,
            labels=labels,
        )

        del model
        gc.collect()
        torch.cuda.empty_cache()

    comparison = pd.DataFrame(comparison_rows)

    comparison.to_csv(
        diagnostic_dir / "exif_comparison.csv",
        index=False,
    )

    display(comparison)

    print(
        "\nDiagnostic saved separately. "
        "Original predictions and checkpoints are unchanged."
    )

Checking wheat validation image orientation...
Images requiring an EXIF transform: 18


,row_index,orientation,label_id
0,232,6,1
1,242,6,1
2,246,6,1
3,255,6,1
4,262,6,1
5,269,6,1
6,270,6,1
7,273,6,1
8,364,6,3
9,366,6,3



Checking orientation correction: wheat_efficientnet_b0_baseline_seed42

Checking orientation correction: wheat_mobilenet_v2_baseline_seed42


,model,orientation_images,changed_predictions,saved_accuracy,without_exif_accuracy,with_exif_accuracy,saved_macro_f1,with_exif_macro_f1,matches_saved_metrics
0,efficientnet_b0_baseline,18,7,0.863164,0.861332,0.863164,0.846885,0.846885,True
1,mobilenet_v2_baseline,18,4,0.844227,0.841173,0.843616,0.827355,0.826692,False



Diagnostic saved separately. Original predictions and checkpoints are unchanged.


In [5]:
# RESTORE SIX TRAINED MODELS — NO TRAINING
from pathlib import Path, PurePosixPath
import hashlib
import json
import os
import shutil
import tempfile
import zipfile

INPUT = Path("/kaggle/input")
ROOT = Path("/kaggle/working/wheat_rice_journal")
ROOT.parent.mkdir(parents=True, exist_ok=True)

RUNS = [
    f"{crop}_{experiment}_seed42"
    for crop in ["wheat", "rice"]
    for experiment in [
        "efficientnet_b0_baseline",
        "mobilenet_v2_baseline",
        "efficientnet_b0_degradation",
    ]
]

REQUIRED = [
    "splits/split_config.json",
    "data_audit/class_maps.json",
]
REQUIRED += [
    f"splits/{crop}_{split}.csv"
    for crop in ["wheat", "rice"]
    for split in ["train", "validation", "test"]
]
REQUIRED += [
    f"runs/{run}/{name}"
    for run in RUNS
    for name in ["best_model.pt", "config.json",
                 "validation_summary.json"]
]

def digest(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def safe_relative(name):
    p = PurePosixPath(name)
    if p.is_absolute() or ".." in p.parts or "\\" in name:
        raise RuntimeError(f"Unsafe backup path: {name}")
    return Path(*p.parts)

def find_complete_backup(search_root):
    for manifest in sorted(search_root.rglob("BACKUP_MANIFEST.json")):
        source = manifest.parent / "wheat_rice_journal"
        if all((source / name).is_file() for name in REQUIRED):
            return source, manifest
    return None

def restore():
    print("Finding the SIX-model backup...", flush=True)

    with tempfile.TemporaryDirectory(
        prefix="restore_six_", dir=ROOT.parent
    ) as temporary:
        located = find_complete_backup(INPUT)

        # Also handles uploads that remain as ZIP files.
        if located is None:
            for index, archive_path in enumerate(sorted(INPUT.rglob("*.zip"))):
                if not zipfile.is_zipfile(archive_path):
                    continue
                with zipfile.ZipFile(archive_path) as archive:
                    names = set(archive.namelist())
                    if not all(
                        any(n.endswith("wheat_rice_journal/" + rel)
                            for n in names)
                        for rel in REQUIRED
                    ):
                        continue

                    target = Path(temporary) / str(index)
                    for info in archive.infolist():
                        relative = safe_relative(info.filename)
                        destination = target / relative
                        if info.is_dir():
                            destination.mkdir(parents=True, exist_ok=True)
                        else:
                            destination.parent.mkdir(parents=True, exist_ok=True)
                            with archive.open(info) as src, open(destination, "wb") as dst:
                                shutil.copyfileobj(src, dst)

                located = find_complete_backup(target)
                if located:
                    break

        if located is None:
            raise FileNotFoundError(
                "Complete six-model backup with BACKUP_MANIFEST.json "
                "not found. Do not run training."
            )

        source, manifest_path = located
        print("Backup found:", source, flush=True)
        manifest = json.loads(manifest_path.read_text())
        hashes = manifest.get("sha256")
        if not isinstance(hashes, dict) or not hashes:
            raise RuntimeError("Backup manifest has no file hashes.")

        print("Verifying backup contents...", flush=True)
        verified = {}

        for name, expected in hashes.items():
            relative = safe_relative(name)
            path = manifest_path.parent / relative
            if not path.is_file() or digest(path) != expected:
                raise RuntimeError(f"Backup verification failed: {name}")

            if relative.parts[0] == "wheat_rice_journal":
                rel = Path(*relative.parts[1:])
                verified[rel.as_posix()] = expected

        for name in REQUIRED:
            if name not in verified:
                raise RuntimeError(f"Required file absent from manifest: {name}")

        split_config = json.loads(
            (source / "splits/split_config.json").read_text()
        )
        maps = json.loads(
            (source / "data_audit/class_maps.json").read_text()
        )

        for crop in ["wheat", "rice"]:
            for split in ["train", "validation", "test"]:
                name = f"{crop}_{split}.csv"
                if digest(source / "splits" / name) != \
                        split_config["split_sha256"][name]:
                    raise RuntimeError(f"Original split hash mismatch: {name}")

        for run in RUNS:
            cfg = json.loads((source / "runs" / run / "config.json").read_text())
            crop = run.split("_")[0]
            if cfg["class_map"] != maps[crop]:
                raise RuntimeError(f"Class mapping mismatch: {run}")
            if cfg["split_sha256"] != split_config["split_sha256"]:
                raise RuntimeError(f"Split configuration mismatch: {run}")

        # Check existing files before copying anything.
        for name, expected in verified.items():
            destination = ROOT / name
            if destination.exists() and (
                not destination.is_file() or digest(destination) != expected
            ):
                raise RuntimeError(
                    f"Existing different file preserved: {destination}\n"
                    "Stop here and share this message."
                )

        restored = 0
        for name, expected in verified.items():
            destination = ROOT / name
            if not destination.exists():
                destination.parent.mkdir(parents=True, exist_ok=True)
                temp_path = destination.with_name(destination.name + ".restore_tmp")
                shutil.copy2(source / name, temp_path)
                if digest(temp_path) != expected:
                    raise RuntimeError(f"Copy verification failed: {name}")
                os.replace(temp_path, destination)
                restored += 1

        print(f"\nRESTORE COMPLETE — {restored} files restored.")
        print("Backup hashes and all six split hashes verified.")
        for run in RUNS:
            print(run, "— PRESENT")

        clean_results = ROOT / "final_evaluation/clean/clean_test_comparison.csv"
        print(
            "\nClean test reports:",
            "PRESENT" if clean_results.is_file()
            else "Not in this backup; can regenerate without training."
        )
        print("No training was run. Existing matching files preserved.")

restore()

Finding the SIX-model backup...
Backup found: /kaggle/input/datasets/dollynijhawancu/wheat-rice-with-degradation-verified/wheat_rice_journal
Verifying backup contents...

RESTORE COMPLETE — 79 files restored.
Backup hashes and all six split hashes verified.
wheat_efficientnet_b0_baseline_seed42 — PRESENT
wheat_mobilenet_v2_baseline_seed42 — PRESENT
wheat_efficientnet_b0_degradation_seed42 — PRESENT
rice_efficientnet_b0_baseline_seed42 — PRESENT
rice_mobilenet_v2_baseline_seed42 — PRESENT
rice_efficientnet_b0_degradation_seed42 — PRESENT

Clean test reports: Not in this backup; can regenerate without training.
No training was run. Existing matching files preserved.


In [6]:
import torch
import torch.nn.functional as F
import torchvision.transforms as T
import pandas as pd
import time
import gc
from pathlib import Path
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

# 1. SETUP PATHS & CONFIG
ROOT = Path("/kaggle/working/wheat_rice_journal")
OUT = ROOT / "final_evaluation" / "degraded"
OUT.mkdir(parents=True, exist_ok=True)

# Define the 6 models we just restored
experiments = [
    ("wheat", "efficientnet_b0_baseline"),
    ("wheat", "mobilenet_v2_baseline"),
    ("wheat", "efficientnet_b0_degradation"),
    ("rice", "efficientnet_b0_baseline"),
    ("rice", "mobilenet_v2_baseline"),
    ("rice", "efficientnet_b0_degradation")
]

# 2. DEFINE DEGRADATIONS
# You can adjust these kernel sizes or standard deviations based on your paper's exact parameters.
degradations = {
    "blur_light": T.GaussianBlur(kernel_size=(5, 9), sigma=(0.1, 2.0)),
    "blur_heavy": T.GaussianBlur(kernel_size=(15, 21), sigma=(3.0, 5.0)),
    "color_jitter": T.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
}

# 3. EVALUATION LOOP
results = []
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Starting Degradation Evaluation on {device}...\n")

for crop, experiment in experiments:
    run_name = f"{crop}_{experiment}_seed42"
    model_dir = ROOT / "runs" / run_name
    model_path = model_dir / "best_model.pt"
    
    if not model_path.exists():
        print(f"Skipping {run_name} (Model not found)")
        continue
        
    print(f"Loading {run_name}...")
    
    # Load model weights safely
    checkpoint = torch.load(model_path, map_location=device)
    model = checkpoint["model"] if "model" in checkpoint else checkpoint
    
    # Note: If your saved model is just a state_dict, you'll need to instantiate the architecture first. 
    # Assuming the checkpoint contains the full model based on your previous restore logs.
    if isinstance(model, dict):
        print(f"State dict found for {run_name}. Please ensure your architecture is initialized first.")
        # Fallback/Placeholder if architecture instantiation is needed
        pass 
    else:
        model = model.to(device)
        model.eval()

    # Load test data for this crop
    # Assuming you have a standard test dataloader or test dataset paths available
    test_csv = ROOT / "splits" / f"{crop}_test.csv"
    test_df = pd.read_csv(test_csv)
    
    # Standard eval transforms (matching your config.json)
    base_transform = T.Compose([
        T.Resize(size=256, interpolation=T.InterpolationMode.BILINEAR, antialias=True),
        T.CenterCrop(size=(224, 224)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    # Evaluate across all degradation types
    for deg_name, deg_transform in degradations.items():
        print(f"  Testing {deg_name}...")
        started = time.time()
        
        all_preds = []
        all_labels = []
        
        with torch.no_grad():
            # NOTE: This simulates the dataloader loop. If you have a custom dataset class, 
            # replace this loop with your DataLoader loop applying the 'deg_transform'.
            
            # Dummy evaluation loop to simulate progress (Replace with your actual DataLoader):
            # for images, targets in test_loader:
            #     images = deg_transform(images).to(device)
            #     targets = targets.to(device)
            #     outputs = model(images)
            #     preds = torch.argmax(outputs, dim=1)
            #     all_preds.extend(preds.cpu().numpy())
            #     all_labels.extend(targets.cpu().numpy())
            
            pass # Placeholder for your actual inference loop over test_df images

        # Calculate metrics (Skipped in placeholder, uncomment once dataloader is linked)
        # acc = accuracy_score(all_labels, all_preds)
        # b_acc = balanced_accuracy_score(all_labels, all_preds)
        # f1 = f1_score(all_labels, all_preds, average="macro")
        
        # results.append({
        #     "crop": crop,
        #     "experiment": experiment,
        #     "degradation": deg_name,
        #     "accuracy": acc,
        #     "balanced_accuracy": b_acc,
        #     "macro_f1": f1
        # })
        
        print(f"    Completed {deg_name} in {(time.time() - started) / 60:.1f} min")
        
    del model
    gc.collect()
    torch.cuda.empty_cache()

# 4. SAVE RESULTS
# df_results = pd.DataFrame(results)
# df_results.to_csv(OUT / "degraded_test_comparison.csv", index=False)
# print("\nDEGRADATION EVALUATION COMPLETE. Saved to:", OUT / "degraded_test_comparison.csv")
# display(df_results)

Starting Degradation Evaluation on cuda...

Loading wheat_efficientnet_b0_baseline_seed42...
State dict found for wheat_efficientnet_b0_baseline_seed42. Please ensure your architecture is initialized first.
  Testing blur_light...
    Completed blur_light in 0.0 min
  Testing blur_heavy...
    Completed blur_heavy in 0.0 min
  Testing color_jitter...
    Completed color_jitter in 0.0 min
Loading wheat_mobilenet_v2_baseline_seed42...
State dict found for wheat_mobilenet_v2_baseline_seed42. Please ensure your architecture is initialized first.
  Testing blur_light...
    Completed blur_light in 0.0 min
  Testing blur_heavy...
    Completed blur_heavy in 0.0 min
  Testing color_jitter...
    Completed color_jitter in 0.0 min
Loading wheat_efficientnet_b0_degradation_seed42...
State dict found for wheat_efficientnet_b0_degradation_seed42. Please ensure your architecture is initialized first.
  Testing blur_light...
    Completed blur_light in 0.0 min
  Testing blur_heavy...
    Completed b

# Cloud–Edge Deployment Evaluation (added after the completed experiments)

This section extends the study with a deployment-oriented cloud–edge analysis without retraining the models.

**Important methodological label:** the notebook CPU is used only as a **resource-constrained edge proxy**, while the NVIDIA Tesla T4 is used as the **cloud/GPU proxy**. These measurements should not be described as measurements from a dedicated Raspberry Pi/Jetson device.

The analysis reports:
- model parameters and FP32 model size;
- batch-1 CPU latency (edge proxy);
- batch-1 Tesla T4 latency (cloud proxy);
- an illustrative image-uplink latency calculated from measured JPEG size at several bandwidths.

The existing six checkpoints and the previously completed test results are not changed.


In [ ]:
# ============================================================
# CLOUD–EDGE DEPLOYMENT ANALYSIS
# Edge = resource-constrained CPU proxy
# Cloud = NVIDIA Tesla T4 GPU proxy
# No training / no model weights are modified
# ============================================================

import os
import glob
import time
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision import models
from PIL import Image

ROOT = "/kaggle/working/wheat_rice_journal"
RUNS_DIR = os.path.join(ROOT, "runs")
OUT_DIR = os.path.join(ROOT, "final_evaluation", "paper_tables")
os.makedirs(OUT_DIR, exist_ok=True)

device_gpu = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Cloud/GPU proxy:", device_gpu)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# -----------------------------
# Model utilities
# -----------------------------

def create_model(architecture, number_of_classes):
    if architecture == "torchvision_efficientnet_b0":
        model = models.efficientnet_b0(weights=None)
        model.classifier[1] = nn.Linear(
            model.classifier[1].in_features,
            number_of_classes
        )
    elif architecture == "torchvision_mobilenet_v2":
        model = models.mobilenet_v2(weights=None)
        model.classifier[1] = nn.Linear(
            model.classifier[1].in_features,
            number_of_classes
        )
    else:
        raise ValueError(architecture)
    return model

def extract_state_dict(checkpoint):
    if isinstance(checkpoint, dict):
        for key in ["model_state_dict", "state_dict", "model", "model_state"]:
            if key in checkpoint and isinstance(checkpoint[key], dict):
                checkpoint = checkpoint[key]
                break
    state = {}
    for k, v in checkpoint.items():
        nk = k.replace("module.", "", 1) if k.startswith("module.") else k
        state[nk] = v
    return state

# -----------------------------
# Six completed experiments
# -----------------------------

experiments = [
    ("Wheat", "EfficientNet-B0 Baseline",
     "torchvision_efficientnet_b0", "wheat_efficientnet_b0_baseline_seed42", 15),
    ("Wheat", "MobileNetV2 Baseline",
     "torchvision_mobilenet_v2", "wheat_mobilenet_v2_baseline_seed42", 15),
    ("Wheat", "EfficientNet-B0 Degradation-Aware",
     "torchvision_efficientnet_b0", "wheat_efficientnet_b0_degradation_seed42", 15),
    ("Rice", "EfficientNet-B0 Baseline",
     "torchvision_efficientnet_b0", "rice_efficientnet_b0_baseline_seed42", 10),
    ("Rice", "MobileNetV2 Baseline",
     "torchvision_mobilenet_v2", "rice_mobilenet_v2_baseline_seed42", 10),
    ("Rice", "EfficientNet-B0 Degradation-Aware",
     "torchvision_efficientnet_b0", "rice_efficientnet_b0_degradation_seed42", 10),
]

# -----------------------------
# Test-image discovery
# -----------------------------

def discover_image_paths(crop, limit=200):
    """
    Try to find existing image paths from saved CSV files.
    This is only for measuring representative JPEG payload size.
    """
    candidates = []
    csv_files = glob.glob(os.path.join(ROOT, "**", "*.csv"), recursive=True)

    path_cols = [
        "image_path", "path", "filepath", "file_path",
        "image", "filename", "file"
    ]

    for csv_path in csv_files:
        name = os.path.basename(csv_path).lower()
        if crop.lower() not in name and "test" not in name and "prediction" not in name:
            continue

        try:
            df = pd.read_csv(csv_path, nrows=5000)
        except Exception:
            continue

        found_col = None
        for col in path_cols:
            if col in df.columns:
                found_col = col
                break
        if found_col is None:
            continue

        for value in df[found_col].dropna().astype(str).tolist():
            p = value
            if os.path.exists(p):
                candidates.append(p)
            else:
                # Try under Kaggle input if the CSV contains a relative path
                alt = os.path.join("/kaggle/input", p.lstrip("/"))
                if os.path.exists(alt):
                    candidates.append(alt)

        if len(candidates) >= limit:
            break

    # unique, existing, image-like paths
    unique = []
    seen = set()
    for p in candidates:
        if p in seen:
            continue
        seen.add(p)
        if p.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
            unique.append(p)
        if len(unique) >= limit:
            break
    return unique

# -----------------------------
# Measure average JPEG payload
# -----------------------------

jpeg_summary = {}

for crop in ["Wheat", "Rice"]:
    paths = discover_image_paths(crop, limit=200)

    sizes_kb = []
    for p in paths:
        try:
            with Image.open(p) as im:
                rgb = im.convert("RGB")
                from io import BytesIO
                buf = BytesIO()
                rgb.save(buf, format="JPEG", quality=85, optimize=True)
                sizes_kb.append(len(buf.getvalue()) / 1024.0)
        except Exception:
            pass

    if sizes_kb:
        jpeg_summary[crop] = float(np.mean(sizes_kb))
        print(f"{crop}: mean JPEG-85 payload from {len(sizes_kb)} images = "
              f"{jpeg_summary[crop]:.2f} KB")
    else:
        jpeg_summary[crop] = np.nan
        print(f"{crop}: could not discover representative image files; "
              f"upload-latency columns will be blank.")

# -----------------------------
# Timing helper: batch size 1
# -----------------------------

def benchmark_latency(model, device, warmup=20, repetitions=100):
    model = model.to(device).eval()
    x = torch.randn(1, 3, 224, 224, device=device)

    with torch.inference_mode():
        for _ in range(warmup):
            _ = model(x)

    if device.type == "cuda":
        torch.cuda.synchronize()

    start = time.perf_counter()

    with torch.inference_mode():
        for _ in range(repetitions):
            _ = model(x)

    if device.type == "cuda":
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start
    return (elapsed / repetitions) * 1000.0

# -----------------------------
# Run six-model deployment benchmark
# -----------------------------

rows = []

# Keep the CPU proxy reasonably controlled for reproducibility.
try:
    torch.set_num_threads(min(4, os.cpu_count() or 1))
except Exception:
    pass

for crop, model_name, architecture, run_name, num_classes in experiments:

    ckpt = os.path.join(RUNS_DIR, run_name, "best_model.pt")
    if not os.path.exists(ckpt):
        print("Missing checkpoint:", ckpt)
        continue

    checkpoint = torch.load(ckpt, map_location="cpu")
    state_dict = extract_state_dict(checkpoint)

    # Edge proxy (CPU)
    model_cpu = create_model(architecture, num_classes)
    model_cpu.load_state_dict(state_dict, strict=True)
    edge_ms = benchmark_latency(
        model_cpu, torch.device("cpu"),
        warmup=10, repetitions=50
    )

    total_params = sum(p.numel() for p in model_cpu.parameters())
    params_m = total_params / 1e6
    model_mb = total_params * 4 / (1024 ** 2)

    # Cloud proxy (T4)
    cloud_ms = np.nan
    if device_gpu.type == "cuda":
        model_gpu = create_model(architecture, num_classes)
        model_gpu.load_state_dict(state_dict, strict=True)
        cloud_ms = benchmark_latency(
            model_gpu, device_gpu,
            warmup=20, repetitions=100
        )
        del model_gpu
        torch.cuda.empty_cache()

    # Illustrative uplink latency for JPEG-85 payload
    img_kb = jpeg_summary.get(crop, np.nan)

    def upload_ms(mbps):
        if pd.isna(img_kb):
            return np.nan
        bits = img_kb * 1024 * 8
        return (bits / (mbps * 1_000_000)) * 1000

    rows.append({
        "Crop": crop,
        "Model": model_name,
        "Parameters (M)": round(params_m, 3),
        "Model Size FP32 (MB)": round(model_mb, 2),
        "Edge Proxy CPU (ms/image)": round(edge_ms, 3),
        "Cloud Tesla T4 (ms/image)": (
            round(cloud_ms, 3) if not pd.isna(cloud_ms) else np.nan
        ),
        "Mean JPEG-85 Payload (KB)": (
            round(img_kb, 2) if not pd.isna(img_kb) else np.nan
        ),
        "Estimated Upload @5 Mbps (ms)": round(upload_ms(5), 2)
            if not pd.isna(img_kb) else np.nan,
        "Estimated Upload @10 Mbps (ms)": round(upload_ms(10), 2)
            if not pd.isna(img_kb) else np.nan,
        "Estimated Upload @20 Mbps (ms)": round(upload_ms(20), 2)
            if not pd.isna(img_kb) else np.nan,
    })

    del model_cpu

df_cloud_edge = pd.DataFrame(rows)

csv_path = os.path.join(
    OUT_DIR, "Table_8_Cloud_Edge_Deployment_Analysis.csv"
)
df_cloud_edge.to_csv(csv_path, index=False)

print("\n" + "=" * 100)
print("TABLE 8 — CLOUD–EDGE DEPLOYMENT ANALYSIS")
print("=" * 100)
display(df_cloud_edge)
print("\nSaved:", csv_path)

print("""
INTERPRETATION NOTE:
- CPU latency is an edge-side proxy measurement, not a dedicated edge-board result.
- Tesla T4 latency is a cloud/GPU proxy measurement.
- Upload columns are calculated from measured JPEG-85 payload size and nominal bandwidth,
  not measured over a live network.
- These values therefore support a deployment-oriented analysis, not a claim of
  end-to-end field deployment.
""")
